In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
karkavelrajaj_amazon_sales_dataset_path = kagglehub.dataset_download('karkavelrajaj/amazon-sales-dataset')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

pd.set_option('display.max_columns', None)
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# Load the dataset for understanding and analysis
data = pd.read_csv(r'/kaggle/input/datasets/karkavelrajaj/amazon-sales-dataset/amazon.csv')
df = pd.DataFrame(data)
df.head()

In [ ]:
df.columns

In [ ]:
df.info() # Check on the datatypes of the columns in the dataset as well as the column counts

In [ ]:
#df.isna().sum()
#df = df.dropna()
df = df.dropna(subset=['rating','rating_count'])
df.isna().sum()

In [ ]:
# From above, there are no missing values, and the values have been registered in string datatypes
# Will convert the necessary columns to their respective data types
# Clean 'discounted_price' and 'actual_price' using the catch-all regex
df['discounted_price'] = df['discounted_price'].astype(str).str.replace(r'[^0-9.]', '', regex=True)
df['actual_price'] = df['actual_price'].astype(str).str.replace(r'[^0-9.]', '', regex=True)

# Clean 'discount_percentage' safely (escaping the regex | character as \|)
df['discount_percentage'] = df['discount_percentage'].astype(str).str.replace(r'[\%|]', '', regex=True)

# Clean 'rating_count' and 'rating' safely
df['rating_count'] = df['rating_count'].astype(str).str.replace(r'[^0-9.]', '', regex=True)
df['rating'] = df['rating'].astype(str).str.replace(r'[^0-9.]', '', regex=True)

# Correctly convert multiple columns to float using .apply()
cols_to_numeric = ['discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count']
df[cols_to_numeric] = df[cols_to_numeric].apply(pd.to_numeric, errors='coerce')


In [ ]:
df.describe() # Now let's see what's in the numeric columns we've converted

## Exploratory Data Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Initialize a 1-row, 2-column figure layout window
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 2. Place the Histplot directly onto the first panel axis (ax1)
sns.histplot(df['discounted_price'], bins=30, kde=True, color='blue', ax=ax1)
ax1.set_title('Distribution of Discounted Prices')
ax1.set_xlabel('Price')

# 3. Place the Heatmap directly onto the second panel axis (ax2)
sns.heatmap(df[cols_to_numeric].corr(), annot=True, cmap='coolwarm', fmt=".2f", ax=ax2)
ax2.set_title('Correlation Matrix of Sales Features')

# 4. Automatically balance column spacing patterns to avoid label overlapping
plt.tight_layout()
plt.show()


## Machine learning model for prediction

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

# 1. Clean the 'category' text column (taking the first main category before the pipe '|')
df['main_category'] = df['category'].astype(str).apply(lambda x: x.split('|')[0].strip())

# 2. Extract numeric features and One-Hot Encode the main categories
X_numeric = df[['actual_price', 'rating', 'rating_count']]
X_categorical = pd.get_dummies(df['main_category'], drop_first=True)

# Combine them into a single Feature DataFrame
X = pd.concat([X_numeric, X_categorical], axis=1)
y = df['discounted_price']

# 3. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. Train and evaluation model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

print(f"New R2 Score: {model.score(X_test_scaled, y_test):.4f}")


In [ ]:
predicted = pd.DataFrame(model.predict(X_test))
predicted.tail(15)
